In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
comp = pd.read_csv("data/compustat.csv")
macro = pd.read_csv("data/crisp treasury and inflation.csv")

# Convert to datetime
comp['datadate'] = pd.to_datetime(comp['datadate'])
macro['caldt'] = pd.to_datetime(macro['caldt'])

# Create year-quarter key
comp['yearq'] = comp['datadate'].dt.to_period('Q')
macro['yearq'] = macro['caldt'].dt.to_period('Q')



In [15]:
macro_q = (
    macro
    .groupby('yearq')
    .agg({
        'b30ret': 'mean',
        'b20ret': 'mean',
        'b10ret': 'mean',
        'b5ret':  'mean',
        'b1ret':  'mean',
        't90ret': 'mean',
        't30ret': 'mean',
        'cpiret': 'mean',

        'b30ind': 'last',
        'b20ind': 'last',
        'b10ind': 'last',
        'b5ind':  'last',
        'b1ind':  'last',
        't90ind': 'last',
        't30ind': 'last',
        'cpiind': 'last'
    })
    .reset_index()
)


In [ ]:
df = comp.merge(
    macro_q,
    on='yearq',
    how='left'
)

In [17]:
# Sort properly
df = df.sort_values(['gvkey', 'yearq'])

# Next-quarter liquidity
df['wcapq_fwd'] = df.groupby('gvkey')['wcapq'].shift(-1)


In [18]:
features = [
    'atq', 'mkvaltq', 'cheq', 'actq', 'lctq',
    'ltq', 'dlttq', 'dlcq',
    'niq', 'oibdpq', 'saleq',
    'DAYMAT', 'YLDMAT'
]

df_model = df[features + ['wcapq_fwd', 'fyearq']].dropna()

X = df_model[features]
y = df_model['wcapq_fwd']


In [19]:
X_train = X[df_model['fyearq'] <= 2018]
X_test  = X[df_model['fyearq'] > 2018]

y_train = y.loc[X_train.index]
y_test  = y.loc[X_test.index]


In [20]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)


In [21]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")


RMSE: 1508.9254
R²:   0.7305


In [22]:
imp = pd.Series(rf.feature_importances_, index=features)
print(imp.sort_values(ascending=False))

cheq       8.359562e-01
lctq       5.765161e-02
actq       5.236425e-02
saleq      2.811173e-02
dlcq       1.046660e-02
atq        7.207630e-03
ltq        5.009369e-03
mkvaltq    1.848131e-03
dlttq      6.432298e-04
oibdpq     3.137532e-04
YLDMAT     2.961994e-04
niq        1.311402e-04
DAYMAT     1.723886e-07
dtype: float64
